<a href="https://colab.research.google.com/github/Ishika-S27/NNDL_LAB/blob/main/NNDL_prac5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q importlib_resources

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import MobileNetV2

import tensorflow_datasets as tfds

(train_ds, validation_ds), dataset_info = tfds.load(
    "tf_flowers",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True
)

class_names = dataset_info.features["label"].names

print("Classes:", class_names)

plt.figure(figsize=(8, 8))

for i, (image, label) in enumerate(train_ds.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()])
    plt.axis("off")

plt.tight_layout()
plt.show()

IMG_SIZE = (160, 160)

train_ds = train_ds.map(
    lambda x, y: (tf.image.resize(x, IMG_SIZE) / 255.0, y)
)

validation_ds = validation_ds.map(
    lambda x, y: (tf.image.resize(x, IMG_SIZE) / 255.0, y)
)

train_ds = train_ds.batch(32).prefetch(tf.data.AUTOTUNE)
validation_ds = validation_ds.batch(32).prefetch(tf.data.AUTOTUNE)

base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=5
)

loss, accuracy = model.evaluate(validation_ds)

print("Validation Loss =", loss)
print("Validation Accuracy =", accuracy)

for images, labels in validation_ds.take(1):
    prediction = model.predict(images)

    predicted = np.argmax(prediction[0])
    actual = labels[0].numpy()

    print("Predicted:", class_names[predicted])
    print("Actual:", class_names[actual])

    plt.imshow(images[0])
    plt.title(
        f"Predicted: {class_names[predicted]}\n"
        f"Actual: {class_names[actual]}"
    )
    plt.axis("off")
    plt.show()

plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()